# Unificar y limpiar la base de rasgos funcionales de Galeras

Parte de 2 archivos:
- **`03__Rasgos_Galeras.xlsx`** (hoja `Hoja 1`): **base principal**, 78 árboles, campaña de campo más reciente (incluye área foliar escaneada y SLA ya calculado).
- **`01__Rasgos_Galeras_sept.xlsx`** (hoja `Galeras`): **base de contraste**, 625 filas (todo el listado de árboles de Galeras), de las cuales 68 tienen algún dato de rasgos medido. Se usa para verificar la base principal y rescatar mediciones que no llegaron a incorporarse en `03`.

**Qué hace el notebook, en orden:**
1. Extrae qué celdas estaban resaltadas en **amarillo** en cada archivo original (para poder conservarlas).
2. Estandariza nombres de columnas de ambas bases para que se parezcan a la estructura ya usada en la base de rasgos de Sumaco (`Rasgos_2025`).
3. Cruza el `treeID` viejo (de `01`) con el `new_tree_ID_2025` (de `03`) usando la columna `new ID 2024` de `01`, que resultó ser el mismo espacio de IDs que `TreeID_2025` de `03`.
4. Identifica árboles con datos de rasgos en `01` que **no están** en la base principal `03` — información potencialmente faltante.
5. Une `03` (principal) + esas filas exclusivas de `01`.
6. Limpia valores de texto/errores de captura (longitudes con varios fragmentos, fechas mal interpretadas por Excel, comentarios mezclados con el número de hojas).
7. Calcula los rasgos funcionales: **wood density, WSG, stem water content, mean leaf thickness, SLA**. (`LDMC` y `force to punch` **no se pudieron calcular** — ver nota más abajo.)
8. Revisa rangos y marca inconsistencias en `QC_flag`.
9. Guarda todo en un Excel, **reaplicando el resaltado amarillo original** sobre las celdas correspondientes de la tabla unificada.

## Nota importante: `LDMC` y `force to punch` no se pueden calcular

A diferencia de la base de Sumaco, **ninguno de los 2 archivos de Galeras tiene una columna de peso fresco de hoja** (`Leaf fresh weight`) — solo está el peso seco. Sin peso fresco no se puede calcular `LDMC` (materia seca / materia fresca). Tampoco hay ninguna columna con la fuerza cruda del punzón/penetrómetro, así que `force to punch (kN/m)` tampoco se puede calcular. Ambas columnas quedan en el resultado final como `NaN`, listas para llenarse si en algún momento se agrega esa medición.

## 1. Librerías

In [30]:
import pandas as pd
import numpy as np
import re
import openpyxl
from openpyxl.styles import PatternFill

pd.set_option('display.max_columns', None)

## 2. Extraer las celdas resaltadas en amarillo de cada archivo original

Antes de tocar los datos, se recorre cada archivo con `openpyxl` y se guarda, por fila y columna, si el color de relleno es amarillo puro (`FFFFFF00`). Esto permite reconstruir el resaltado en la tabla final aunque las columnas cambien de nombre o de posición.

In [31]:
def extraer_resaltado_amarillo(path, sheet_name):
    wb = openpyxl.load_workbook(path)
    ws = wb[sheet_name]
    headers = [c.value for c in next(ws.iter_rows(min_row=1, max_row=1))]
    resaltado = {}  # fila de datos (0-index) -> set(nombre de columna original)
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            fill = cell.fill
            if fill and fill.fgColor and fill.fgColor.rgb == 'FFFFFF00':
                col_name = headers[cell.column - 1]
                resaltado.setdefault(cell.row - 2, set()).add(col_name)
    return resaltado

resaltado_01 = extraer_resaltado_amarillo('01. Rasgos_Galeras_sept.xlsx', 'Galeras')
resaltado_03 = extraer_resaltado_amarillo('03. Rasgos Galeras.xlsx',sheet_name="Hoja 1")
print('Filas con celdas amarillas en 01:', len(resaltado_01))
print('Filas con celdas amarillas en 03:', len(resaltado_03))

Filas con celdas amarillas en 01: 23
Filas con celdas amarillas en 03: 12


## 3. Funciones de limpieza

In [32]:
def limpiar_longitud(v):
    """Convierte 'wet lenght cm' a numero. Cuando vienen varias medidas de texto
    (ej. '1.9-2.7' o '1.3-1.4-1.5') se asume que son fragmentos del mismo barreno
    partido en varios trozos y se SUMAN -- el peso humedo/seco reportado es el de
    todos los fragmentos juntos, asi que el volumen debe usar el largo total
    (se probo promediar primero y daba densidades de madera imposibles >1 g/cm3;
    sumando el largo total, las densidades quedan en rango fisico normal).
    Tambien detecta fechas mal interpretadas por Excel (error de captura)."""
    if pd.isna(v):
        return np.nan, None
    if isinstance(v, (pd.Timestamp,)) or hasattr(v, 'year'):
        return np.nan, 'valor de longitud interpretado como fecha por Excel (dato original a revisar)'
    try:
        return float(v), None
    except (TypeError, ValueError):
        pass
    nums = re.findall(r'\d+\.?\d*', str(v))
    if nums:
        vals = [float(n) for n in nums]
        return sum(vals), f'longitud reportada como {len(vals)} fragmentos ({v}), se sumo el largo total'
    return np.nan, f'valor de longitud no interpretable ({v})'

def a_numero(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan

def limpiar_n_hojas(v):
    """n_leaves deberia ser un numero; si vino con un comentario de campo
    mezclado en la misma celda, se toma el primer numero."""
    if pd.isna(v):
        return np.nan, None
    if isinstance(v, (int, float)):
        return float(v), None
    m = re.match(r'^\s*(\d+)', str(v))
    if m:
        return float(m.group(1)), f'n_leaves traia texto mezclado ({v}), se tomo el primer numero'
    return np.nan, f'n_leaves no interpretable ({v})'

## 4. Cargar y estandarizar `03` (base principal)

Se renombran las columnas para que se parezcan a `Rasgos_2025` de Sumaco (`leaf_dry_weight_g`, `wet_wood_weight_g`, etc.). `mean_thickness_(mm)` y `SLA` originales se guardan aparte (`SLA_original`) y se recalculan desde cero más abajo, como control cruzado.

In [33]:
df3 = pd.read_excel('03. Rasgos Galeras.xlsx', sheet_name='Hoja 1')

ren3 = {
    'Site': 'Site', 'Plot_code': 'PlotID', 'Sub': 'Subplot', 'Date ': 'sampling_date',
    'TreeID_2025': 'new_tree_ID_2025', 'Family': 'family', 'genus': 'genus', 'specie': 'species',
    'altitude': 'altitude_m',
    'leaf 1, thickn 1': 'leaf1_thickness1', 'leaf 1, thickn 2': 'leaf1_thickness2', 'leaf 1, thickn 3': 'leaf1_thickness3',
    'leaf 2, thickn 1': 'leaf2_thickness1', 'leaf 2, thickn 2': 'leaf2_thickness2', 'leaf 2, thickn 3': 'leaf2_thickness3',
    'leaf 3, thickn 1': 'leaf3_thickness1', 'leaf 3, thickn 2': 'leaf3_thickness2', 'leaf 3, thickn 3': 'leaf3_thickness3',
    '# leaves scan': 'n_leaves', 'compound': 'compound_leaf_note',
    'Leaf dry weight (g)': 'leaf_dry_weight_g', 'Leaf area (cm²)': 'leaf_area_cm2', 'SLA': 'SLA_original',
    'Comment': 'comment', 'wood characteristics': 'wood_characteristics',
    'crust thickness': 'crust_thickness_cm', 'wet lenght cm': 'wet_length_cm_raw',
    'wet wood weight': 'wet_wood_weight_g', 'dry wood weight': 'dry_wood_weight_g',
}
df3 = df3.rename(columns=ren3)
df3 = df3.drop(columns=['#', 'mean_thickness_(mm)', 'Unnamed: 25'], errors='ignore')
df3['Plot'] = df3['PlotID'].str.extract(r'(\d+)').astype(float)
df3['treeID'] = np.nan  # se completa mas abajo con el cruce contra 01
df3['source_file'] = '03__Rasgos_Galeras.xlsx (principal)'
df3['_orig_row'] = range(len(df3))
df3.shape

(78, 33)

## 5. Cargar y estandarizar `01` (contraste)

`new ID 2024` se renombra directamente a `new_tree_ID_2025`: se verificó que es el mismo espacio de identificadores que `TreeID_2025` de `03` (56 árboles coinciden exactamente, con los mismos valores de peso seco de hoja y madera). El texto libre de `comp. leaf` (ej. "con latex", "flores") se traslada a `comment`, ya que no es un dato estructurado de hoja compuesta.

In [34]:
df1 = pd.read_excel('01. Rasgos_Galeras_sept.xlsx', sheet_name='Galeras')

ren1 = {
    'Plot': 'Plot', 'site': 'Site', 'PlotID': 'PlotID', 'Subplot_ID': 'Subplot', 'treeID': 'treeID',
    'family': 'family', 'genus': 'genus', 'species': 'species', 'sampling date': 'sampling_date',
    'new ID 2024': 'new_tree_ID_2025',
    'comp. leaf': 'comment',
    'leaf 1 med 1': 'leaf1_thickness1', 'leaf 1 med 2': 'leaf1_thickness2', 'leaf 1 med 3': 'leaf1_thickness3',
    'leaf 2 med 1': 'leaf2_thickness1', 'leaf 2 med 2': 'leaf2_thickness2', 'leaf 2 med 3': 'leaf2_thickness3',
    'leaf 3 med 1': 'leaf3_thickness1', 'leaf 3 med 2': 'leaf3_thickness2', 'leaf 3 med 3': 'leaf3_thickness3',
    'Leaf dry weight (g)': 'leaf_dry_weight_g', '# leaves': 'n_leaves',
    'wood characteristics': 'wood_characteristics', 'crust thickness': 'crust_thickness_cm',
    'wet lenght cm': 'wet_length_cm_raw', 'wet wood weight': 'wet_wood_weight_g', 'dry wood weight': 'dry_wood_weight_g',
}
df1 = df1.rename(columns=ren1)
df1 = df1.drop(columns=['Select_2024', 'Colecct_2024', 'new entry 2024', 'Espesor'], errors='ignore')
df1['source_file'] = '01__Rasgos_Galeras_sept.xlsx (contraste)'
df1['_orig_row'] = range(len(df1))
df1.shape

(625, 29)

## 6. Cruzar `treeID` (viejo) con `new_tree_ID_2025`

Se construye un diccionario `new_tree_ID_2025 -> treeID` a partir de `01`, y se usa para completar el `treeID` viejo en la base principal `03` (que no lo trae). De paso se detectan valores de `new ID 2024` duplicados dentro de `01` (mismo ID asignado a 2 filas distintas): son un error de captura a revisar.

In [35]:
crosswalk = (df1[df1['new_tree_ID_2025'].notna()]
             .drop_duplicates('new_tree_ID_2025')
             .set_index('new_tree_ID_2025')['treeID'])

dup_new_id = df1['new_tree_ID_2025'].value_counts()
ids_duplicados = dup_new_id[dup_new_id > 1].index.tolist()
print("'new ID 2024' duplicado en 01 (mismo ID en >1 fila):", ids_duplicados)

df3['treeID'] = df3['new_tree_ID_2025'].map(crosswalk)

'new ID 2024' duplicado en 01 (mismo ID en >1 fila): [8732.0, 8721.0]


## 7. Rescatar de `01` las filas con dato real que faltan en `03`

De las 68 filas de `01` con algún dato de rasgos medido, 56 ya están representadas en `03` (con los mismos valores exactos — confirma que son la misma medición). Las 12 restantes tienen datos de madera que **no llegaron a la base principal** — se agregan a la base unificada, marcadas para que las confirmes.

In [36]:
set_03 = set(df3['new_tree_ID_2025'].dropna().astype(int))
tiene_dato_01 = df1['leaf_dry_weight_g'].notna() | df1['dry_wood_weight_g'].notna()
df1_con_dato = df1[tiene_dato_01].copy()
df1_con_dato['en_03'] = df1_con_dato['new_tree_ID_2025'].apply(
    lambda v: (int(v) in set_03) if pd.notna(v) else False)

faltantes_en_03 = df1_con_dato[~df1_con_dato['en_03']].drop(columns=['en_03']).copy()
faltantes_en_03['QC_flag'] = 'Presente solo en base secundaria (01); posible informacion faltante en 03; '
print('Filas de 01 con dato de rasgos ausentes en 03:', len(faltantes_en_03))
faltantes_en_03[['PlotID', 'treeID', 'new_tree_ID_2025', 'family', 'genus', 'species']]

Filas de 01 con dato de rasgos ausentes en 03: 12


,PlotID,treeID,new_tree_ID_2025,family,genus,species
94,GAL_22,NaN,8967.0,Myristicaceae,Compsoneura,capitellata
112,GAL_23,569,8997.0,Lecythidaceae,Grias,peruviana
114,GAL_23,571,8996.0,Rubiaceae,Posoqueria,latifolia
119,GAL_23,576,8990.0,Rubiaceae,Kutchubaea,semisericea
239,GAL_28,716,8736.0,Chrysobalanaceae,Licania,macrocarpa
422,GAL_50,1484,8618.0,Sapotaceae,Sapo28,NaN
427,GAL_50,1456,8788.0,Symplocaceae,Symplocos,cf spruceana
429,GAL_50,1458,8794.0,Elaeocarpaceae,Sloanea,NaN
452,GAL_50,1483,8617.0,Myrtaceae,Siphoneugena,densiflora
494,GAL_60,1810,8770.0,Lauraceae,Ocotea,NaN


## 8. Unificar `03` + filas exclusivas de `01`

In [37]:
unificado = pd.concat([df3, faltantes_en_03], ignore_index=True, sort=False)
unificado['QC_flag'] = unificado['QC_flag'].fillna('')
print('Base unificada:', unificado.shape)

Base unificada: (90, 34)


## 9. Limpiar longitudes de madera y número de hojas

In [38]:
limpio = unificado['wet_length_cm_raw'].apply(limpiar_longitud)
unificado['wet_length_cm'] = limpio.apply(lambda t: t[0])
notas_longitud = limpio.apply(lambda t: t[1])
mask_nota = notas_longitud.notna()
unificado.loc[mask_nota, 'QC_flag'] += notas_longitud[mask_nota] + '; '

for c in ['wet_wood_weight_g', 'dry_wood_weight_g', 'crust_thickness_cm', 'leaf_dry_weight_g', 'leaf_area_cm2']:
    unificado[c] = unificado[c].apply(a_numero)

limpio_hojas = unificado['n_leaves'].apply(limpiar_n_hojas)
unificado['n_leaves'] = limpio_hojas.apply(lambda t: t[0])
notas_hojas = limpio_hojas.apply(lambda t: t[1])
mask_nota_h = notas_hojas.notna()
unificado.loc[mask_nota_h, 'QC_flag'] += notas_hojas[mask_nota_h] + '; '

# 9.2 Agregar LA que faltan

In [39]:
LA = pd.read_excel("Galeras_FM.xlsx")
#LA["leaf_area_cm2"]=LA["sum"]
unificado.head(2)

,Site,PlotID,Subplot,sampling_date,new_tree_ID_2025,family,genus,species,altitude_m,leaf1_thickness1,leaf1_thickness2,leaf1_thickness3,leaf2_thickness1,leaf2_thickness2,leaf2_thickness3,leaf3_thickness1,leaf3_thickness2,leaf3_thickness3,n_leaves,compound_leaf_note,leaf_dry_weight_g,leaf_area_cm2,SLA_original,comment,wood_characteristics,crust_thickness_cm,wet_length_cm_raw,wet_wood_weight_g,dry_wood_weight_g,Plot,treeID,source_file,_orig_row,QC_flag,wet_length_cm
0,Galeras,GAL_24,A,2025-01-08 00:00:00,8633.0,Clusiaceae,Garcinia,madruno,1450.0,0.300,0.246,0.239,0.251,0.244,0.223,0.158,0.143,0.158,20.0,NaN,17.47,1568.870,89.803663,NaN,NaN,0.1,6.4,1.66,0.86,24.0,NaN,03__Rasgos_Galeras.xlsx (principal),0,,6.4
1,Galeras,GAL_24,B,2025-01-08 00:00:00,8639.0,Lacistemataceae,Lozania,klugii,1450.0,0.129,0.106,0.074,0.099,0.074,0.064,0.092,0.098,0.065,20.0,NaN,10.22,1594.462,156.013894,NaN,NaN,0.3,8.1,2.02,0.84,24.0,NaN,03__Rasgos_Galeras.xlsx (principal),1,,8.1


In [40]:
unificado= pd.merge(unificado, LA, left_on="new_tree_ID_2025", right_on="TreeID", how = "outer")#verificar primero
unificado["leaf_area_cm2"] = unificado["leaf_area_cm2"].combine_first(unificado["sum"])
unificado.to_excel("test.xlsx")

## 10. Calcular los rasgos funcionales

Mismas fórmulas usadas para Sumaco (barrenador de 0.5 cm de diámetro → radio 0.25 cm):

- **Wood density (g/cm³)** = peso seco madera / (π × 0.25² × largo total)
- **WSG** = wood density / densidad del agua (1 g/cm³)
- **Stem water content (%)** = (peso húmedo − peso seco) / peso seco × 100
- **Mean leaf thickness (mm)** = promedio de las 9 mediciones (3 hojas × 3 puntos — aquí Galeras midió 3 puntos por hoja en vez de los 2 que usó Sumaco)
- **SLA (cm²/g)** = área foliar / peso seco de hoja (solo disponible para las filas que tienen área escaneada, prácticamente todas de `03`)
- **LDMC** y **force to punch**: no calculables (ver nota al inicio).

In [41]:
radio_barrenador_cm = 0.5 / 2  # 0.25 cm

unificado['wood_volume_cm3'] = np.pi * radio_barrenador_cm**2 * unificado['wet_length_cm']
unificado['wood_density_g_cm3'] = unificado['dry_wood_weight_g'] / unificado['wood_volume_cm3']
unificado['WSG'] = unificado['wood_density_g_cm3'] / 1.0

unificado['stem_water_content_pct'] = (
    (unificado['wet_wood_weight_g'] - unificado['dry_wood_weight_g']) / unificado['dry_wood_weight_g'] * 100
)

grosor_cols = ['leaf1_thickness1', 'leaf1_thickness2', 'leaf1_thickness3',
               'leaf2_thickness1', 'leaf2_thickness2', 'leaf2_thickness3',
               'leaf3_thickness1', 'leaf3_thickness2', 'leaf3_thickness3']
unificado['mean_leaf_thickness_mm'] = unificado[grosor_cols].mean(axis=1)

unificado['SLA_cm2_g'] = unificado['leaf_area_cm2'] / unificado['leaf_dry_weight_g']

# Guarda de plausibilidad: un area de hoja fuera de rango fisico es un error de captura
# (se encontro una fila con 642172 cm2 -- un numero de hojas escaneadas corrupto en la
# misma fila sugiere que el escaneo de esa hoja quedo mal guardado). Se excluye del SLA.
area_imposible = unificado['leaf_area_cm2'].notna() & (
    (unificado['leaf_area_cm2'] <= 0) | (unificado['leaf_area_cm2'] > 5000))
unificado.loc[area_imposible, 'QC_flag'] += (
    'leaf_area_cm2 fuera de rango fisico plausible (revisar escaneo original), excluido de SLA; ')
unificado.loc[area_imposible, 'SLA_cm2_g'] = np.nan

unificado['LDMC_mg_g'] = np.nan
unificado['force_to_punch_kN_m'] = np.nan

unificado[['wood_density_g_cm3', 'WSG', 'stem_water_content_pct',
           'mean_leaf_thickness_mm', 'SLA_cm2_g']].describe().T

,count,mean,std,min,25%,50%,75%,max
wood_density_g_cm3,80.0,0.642754,0.136666,0.235644,0.575471,0.652173,0.705179,0.949142
WSG,80.0,0.642754,0.136666,0.235644,0.575471,0.652173,0.705179,0.949142
stem_water_content_pct,82.0,99.358497,38.459859,46.875000,76.858573,92.605378,111.314604,290.322581
mean_leaf_thickness_mm,77.0,0.198696,0.074400,0.077778,0.147778,0.184778,0.239444,0.522556
SLA_cm2_g,66.0,120.716787,51.293494,28.302349,91.748316,108.306455,145.304864,372.657362


## 11. Revisar rangos y marcar inconsistencias

Mismos rangos de referencia usados en Sumaco, más un rango para SLA (típico en árboles tropicales: 20–500 cm²/g).

In [42]:
rangos = {
    'wood_density_g_cm3': (0.15, 1.2),
    'WSG': (0.15, 1.2),
    'stem_water_content_pct': (30, 250),
    'mean_leaf_thickness_mm': (0.05, 1.0),
    'SLA_cm2_g': (20, 500),
}
for col, (lo, hi) in rangos.items():
    fuera = unificado[col].notna() & ((unificado[col] < lo) | (unificado[col] > hi))
    unificado.loc[fuera, 'QC_flag'] += f'{col} fuera de rango [{lo}-{hi}]; '

m = unificado['dry_wood_weight_g'] > unificado['wet_wood_weight_g']
unificado.loc[m, 'QC_flag'] += 'peso seco de madera > peso humedo; '

sin_taxo = unificado['family'].isna() & unificado['genus'].isna() & unificado['species'].isna()
unificado.loc[sin_taxo, 'QC_flag'] += 'sin familia/genero/especie; '

sin_id = unificado['treeID'].isna() & unificado['new_tree_ID_2025'].isna()
unificado.loc[sin_id, 'QC_flag'] += 'sin treeID ni new_tree_ID_2025; '

for iid in ids_duplicados:
    filas = unificado[unificado['new_tree_ID_2025'] == iid]
    unificado.loc[filas.index, 'QC_flag'] += f'new_tree_ID_2025={int(iid)} duplicado en la base secundaria (01); '

print('Total filas marcadas:', (unificado['QC_flag'] != '').sum(), 'de', len(unificado))

Total filas marcadas: 22 de 90


## 12. Reordenar columnas

In [43]:
front = ['Site', 'PlotID', 'Plot', 'Subplot', 'treeID', 'new_tree_ID_2025',
         'family', 'genus', 'species', 'sampling_date', 'altitude_m',
         'n_leaves', 'leaf_dry_weight_g', 'leaf_area_cm2', 'SLA_cm2_g', 'SLA_original',
         'mean_leaf_thickness_mm',
         'leaf1_thickness1', 'leaf1_thickness2', 'leaf1_thickness3',
         'leaf2_thickness1', 'leaf2_thickness2', 'leaf2_thickness3',
         'leaf3_thickness1', 'leaf3_thickness2', 'leaf3_thickness3',
         'wood_characteristics', 'crust_thickness_cm', 'wet_length_cm_raw', 'wet_length_cm',
         'wet_wood_weight_g', 'dry_wood_weight_g', 'wood_volume_cm3',
         'wood_density_g_cm3', 'WSG', 'stem_water_content_pct',
         'LDMC_mg_g', 'force_to_punch_kN_m',
         'compound_leaf_note', 'comment', 'QC_flag', 'source_file', '_orig_row']
otras = [c for c in unificado.columns if c not in front]
unificado = unificado[front + otras]
unificado.shape

KeyError: "['PlotID'] not in index"

## 13. Guardar y volver a aplicar el resaltado amarillo

Se guarda el Excel normalmente con `pandas`, y luego se vuelve a abrir con `openpyxl` para pintar de amarillo cada celda que lo estaba en el archivo de origen correspondiente (usando `_orig_row` y el diccionario de renombrado para ubicar la columna correcta en la tabla nueva).

In [ ]:
out_path = 'Rasgos_Galeras_unificado.xlsx'
reporte = unificado[unificado['QC_flag'] != ''][
    ['Site', 'PlotID', 'treeID', 'new_tree_ID_2025', 'family', 'genus', 'species', 'QC_flag', 'source_file']
]

with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    unificado.to_excel(writer, sheet_name='Rasgos_Galeras_unificado', index=False)
    reporte.to_excel(writer, sheet_name='Filas_a_revisar', index=False)

wb = openpyxl.load_workbook(out_path)
ws = wb['Rasgos_Galeras_unificado']
headers = [c.value for c in next(ws.iter_rows(min_row=1, max_row=1))]
col_idx = {h: i + 1 for i, h in enumerate(headers)}
amarillo = PatternFill(start_color='FFFFFF00', end_color='FFFFFF00', fill_type='solid')

for i, row in unificado.reset_index(drop=True).iterrows():
    origen = row['source_file']
    orig_row = row['_orig_row']
    resaltado_orig = resaltado_03 if origen.startswith('03') else resaltado_01
    cols_resaltadas = resaltado_orig.get(orig_row, set())
    for orig_col in cols_resaltadas:
        nuevo_col = (ren3 if origen.startswith('03') else ren1).get(orig_col, orig_col)
        if nuevo_col in col_idx:
            ws.cell(row=i + 2, column=col_idx[nuevo_col]).fill = amarillo

wb.save(out_path)
print('Guardado con resaltado:', out_path)

Guardado con resaltado: Rasgos_Galeras_unificado.xlsx


## 14. Resumen y sugerencias de revisión manual

- **`LDMC` y `force to punch` no calculables**: ninguno de los 2 archivos trae peso fresco de hoja ni fuerza de punzón. Si esas mediciones existen en otra libreta/archivo, se pueden incorporar con la misma lógica usada para Sumaco.
- **12 árboles con dato de madera solo en la base secundaria (`01`)**, ausentes de `03`: revisar si deben incorporarse formalmente a la base principal (listados en la hoja `Filas_a_revisar`, plots 22, 23, 28, 50, 60, 62).
- **2 valores de `new ID 2024` duplicados en `01`** (asignados a 2 filas distintas cada uno): revisar el archivo original y corregir cuál es el ID correcto.
- **Varias celdas de `# leaves` traían un comentario de campo mezclado** con el número (ej. "20 pesadas, escaneadas 17") — se tomó el primer número, pero conviene revisar si ese es el valor correcto a usar en los cálculos.
- **1 fila con área foliar imposible** (642172 cm²) y el número de hojas corrupto en la misma fila (plot GAL_60, árbol 8744) — revisar el escaneo original de esa hoja.
- **Longitudes de madera con varios fragmentos de texto** (ej. `'1.9-2.7'`, `'1.3-1.4-1.5'`): se sumaron asumiendo que son trozos del mismo barreno; confirmar que esa interpretación es correcta con quien tomó el dato en campo.
- **1 fila con `stem_water_content_pct` fuera de rango** (plot GAL_24, árbol new ID 8649): revisar los pesos húmedo/seco de esa muestra.
- **El resaltado amarillo se conservó** en la tabla final, pero recuerda que su significado original (qué quería decir el equipo de campo al resaltar esas celdas) no está documentado — vale la pena preguntar a quien hizo el resaltado antes de asumir que siempre significa "dato dudoso".